In [ ]:
from libraries import *
from pdex import parallel_differential_expression


In [ ]:
adata = sc.read_h5ad("./../../Data/ComboScreen.h5ad")


In [ ]:
adata = adata[adata.obs[['ASCL1', 'KLF14', 'NEUROD1',
       'NEUROG1', 'NR3C1', 'NTC', 'SIM1', 'TET2', 'TWIST1', 'VSX1', 'ZNF385A',
       'ZNF547', 'ZNF660', 'ZNF776']].sum(axis=1) < 3]

In [ ]:
cols = ['ASCL1', 'KLF14', 'NEUROD1', 'NEUROG1', 'NR3C1', 'NTC', 'SIM1',
        'TET2', 'TWIST1', 'VSX1', 'ZNF385A', 'ZNF547', 'ZNF660', 'ZNF776']

def combine_onehot(row):
    # Select all column names where value == 1
    active = [col for col in cols if row[col] == 1]
    # Join multiple actives with '+', or return 'None' if none are active
    return '+'.join(active) if active else 'None'

adata.obs['perturbation'] = adata.obs[cols].apply(combine_onehot, axis=1)


In [ ]:
adata.obs

In [ ]:
sc.pp.normalize_total(adata, target_sum=20000)
sc.pp.log1p(adata)


In [ ]:
adata.obs["perturbation_time"] = (
    adata.obs["perturbation"].astype(str) + "_" + adata.obs["time_point"].astype(str)
)

In [ ]:
adata_day04 = adata[adata.obs["time_point"] == "day04",]

In [ ]:
adata_day04.obs["perturbation_time"].value_counts()

In [ ]:
degs_day04 = parallel_differential_expression(adata_day04, 
                                    groupby_key="perturbation_time", 
                                    reference="NTC_day04", 
                                    is_log1p=True, 
                                    num_workers=128 )
pd.DataFrame(degs_day04).to_csv("Day04DEGs.csv")

In [ ]:
for elem in adata_day04.obs['final_label'].unique():
    adata_day04_tmp = adata_day04[adata_day04.obs['final_label']==elem,:]
    degs_day04_tmp = parallel_differential_expression(adata_day04_tmp, 
                                        groupby_key="perturbation_time", 
                                        reference="NTC_day04", 
                                        is_log1p=True, 
                                        num_workers=128 )
    pd.DataFrame(degs_day04_tmp).to_csv("Day04DEGs_"+elem+".csv")

In [ ]:
adata_day10 = adata[adata.obs["time_point"] == "day10",]

In [ ]:
k_day4=pd.crosstab(adata_day04.obs['final_label'], adata_day04.obs['perturbation'])
k_day4['Cluster']=k_day4.index
k_day4=pd.melt(k_day4, var_name="variable", id_vars=["Cluster"]  )
k_day4=k_day4.sort_values(
    by=["Cluster", "value"], ascending=False
)
k_day4.columns=["Cluster", "Perturbation", "NumberOfCells"]
k_day4["Cluster"] = [x.replace("-", "_") for x in k_day4["Cluster"] ]

k_day10=pd.crosstab(adata_day10.obs['final_label'], adata_day10.obs['perturbation'])
k_day10['Cluster']=k_day10.index
k_day10=pd.melt(k_day10, var_name="variable", id_vars=["Cluster"]  )
k_day10=k_day10.sort_values(
    by=["Cluster", "value"], ascending=False
)
k_day10.columns=["Cluster", "Perturbation", "NumberOfCells"]
k_day10["Cluster"] = [x.replace("-", "_") for x in k_day10["Cluster"] ]


In [ ]:
adata_day04_control = adata_day04[adata_day04.obs["perturbation"]=="NTC",:]
adata_day04_control.obs["final_label"].value_counts()

In [ ]:
import matplotlib.pyplot as plt

for elem in adata_day04_control.obs["final_label"].unique():
    adata_day04_tmp = adata_day04[adata_day04.obs["final_label"]==elem,:]

    counts = adata_day04_tmp.obs.perturbation.value_counts()

    # Clean up the tick labels
    labels = [x.replace("_day04", "") for x in counts.index]

    plt.figure(figsize=(23,5))
    counts.plot(kind="bar", color="skyblue", edgecolor="black")

    plt.xlabel("Target")
    plt.ylabel("Number of cells")
    plt.title(elem)
    plt.xticks(range(len(labels)), labels, rotation=45, ha="right")
    plt.tight_layout()
    plt.show()



In [ ]:
files_day04 = {
    "Differentiated_1_day04": "Day04DEGs_Differentiated_1.csv",
    "Differentiated_2_day04": "Day04DEGs_Differentiated-2.csv",
    "Intermediate_1_day04":   "Day04DEGs_Intermediate-1.csv",
    "Intermediate_2_day04":   "Day04DEGs_Intermediate-2.csv",
    "Intermediate_3_day04":   "Day04DEGs_Intermediate-3.csv",
    "Neuroendocrine_day04":   "Day04DEGs_Neuroendocrine.csv",
}

files_day10 = {
    "Differentiated_1_day10": "Day10DEGs_Differentiated_1.csv",
    "Differentiated_2_day10": "Day10DEGs_Differentiated-2.csv",
    "Intermediate_1_day10":   "Day10DEGs_Intermediate-1.csv",
    "Intermediate_2_day10":   "Day10DEGs_Intermediate-2.csv",
    "Intermediate_3_day10":   "Day10DEGs_Intermediate-3.csv",
    "Neuroendocrine_day10":   "Day10DEGs_Neuroendocrine.csv",
}
def load_and_clean(path: str, presubstr) -> pd.DataFrame:
    df = pd.read_csv(path, index_col=0)
    df = df.loc[df["fdr"] < 0.05].copy()
    df["target"] = df["target"].str.replace(presubstr, "", regex=False)
    
    return df

degs_day04 = {name: load_and_clean(path,"_day04") for name, path in files_day04.items()}
degs_day10 = {name: load_and_clean(path,"_day10") for name, path in files_day10.items()}


In [ ]:
all_counts = []

for key, df in degs_day04.items():
    tmp = (
        df["target"]
        .value_counts()
        .reset_index()
        .rename(columns={
            "index": "perturbation",
            "target": "n_targets"
        })
    )
    tmp["condition"] = key
    all_counts.append(tmp)

df_all_04 = pd.concat(all_counts, ignore_index=True)
df_all_04["condition"] = df_all_04["condition"].str.replace("_day04","" , regex=False)
df_all_04.columns=["Perturbation", "NumberOfDEGenes", "Cluster"]

df_all_04
df_all_04 = pd.merge(
    df_all_04,
    k_day4,
    on=["Perturbation", "Cluster"],
    how="inner"   # or "left", "right", "outer"
)
df_all_04.to_csv("Day04_NCells_NDEGes_Perturbation_Cluster.csv")


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

g = sns.FacetGrid(
    df_all_04,
    col="Cluster",
    col_wrap=3,        # like facet_wrap(ncol = 3)
    height=4,
    sharex=False,
    sharey=False
)

g.map_dataframe(
    sns.scatterplot,
    x="NumberOfCells",
    y="NumberOfDEGenes",
    s=20
)

g.set_axis_labels("Number of cells", "Number of DE genes")
g.set_titles(col_template="{col_name}")

plt.tight_layout()
plt.show()


In [ ]:
k_day4.Cluster.unique()

In [ ]:
all_counts = []

for key, df in degs_day10.items():
    tmp = (
        df["target"]
        .value_counts()
        .reset_index()
        .rename(columns={
            "index": "perturbation",
            "target": "n_targets"
        })
    )
    tmp["condition"] = key
    all_counts.append(tmp)

df_all_10 = pd.concat(all_counts, ignore_index=True)
df_all_10["condition"] = df_all_10["condition"].str.replace("_day10","" , regex=False)

df_all_10.columns=["Perturbation", "NumberOfDEGenes", "Cluster"]


df_all_10 = pd.merge(
    df_all_10,
    k_day10,
    on=["Perturbation", "Cluster"],
    how="inner"   # or "left", "right", "outer"
)
df_all_10.to_csv("Day10_NCells_NDEGes_Perturbation_Cluster.csv")
df_all_10

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

g = sns.FacetGrid(
    df_all_10,
    col="Cluster",
    col_wrap=3,        # like facet_wrap(ncol = 3)
    height=4,
    sharex=False,
    sharey=False
)

g.map_dataframe(
    sns.scatterplot,
    x="NumberOfCells",
    y="NumberOfDEGenes",
    s=20
)

g.set_axis_labels("Number of cells", "Number of DE genes")
g.set_titles(col_template="{col_name}")

plt.tight_layout()
plt.show()

In [ ]:
for elem in df_all_10.Cluster.unique():
    plt.figure(figsize=(12, 6))
    df_all_10_sgn= df_all_10.loc[(df_all_10.NumberOfCells > 50) & (df_all_10.NumberOfDEGenes > 20) & (df_all_10.Cluster == elem),:]
    sns.barplot(
        data=df_all_10_sgn,
        x="Perturbation",
        y="NumberOfDEGenes"
    )

    plt.xlabel("Perturbation")
    plt.ylabel("Number of DE genes")
    plt.xticks(rotation=90)   # important if many perturbations
    plt.tight_layout()
    plt.title("Day 10 - "+ elem +" perturbations with at least 50 cells")
    plt.show()

In [ ]:
for elem in df_all_04.Cluster.unique():
    plt.figure(figsize=(12, 6))
    df_all_04_sgn= df_all_04.loc[(df_all_04.NumberOfCells > 100) & (df_all_04.NumberOfDEGenes > 20) & (df_all_04.Cluster == elem),:]
    sns.barplot(
        data=df_all_04_sgn,
        x="Perturbation",
        y="NumberOfDEGenes"
    )

    plt.xlabel("Perturbation")
    plt.ylabel("Number of DE genes")
    plt.xticks(rotation=90)   # important if many perturbations
    plt.tight_layout()
    plt.title("Day 4 - "+ elem +" perturbations with at least 50 cells")
    plt.show()

In [ ]:
df_all_10_sgn= df_all_10.loc[(df_all_10.NumberOfCells > 50) & (df_all_10.NumberOfDEGenes > 20),:]
df_all_04_sgn= df_all_04.loc[(df_all_04.NumberOfCells > 50) & (df_all_04.NumberOfDEGenes > 20),:]


In [ ]:
df_all_04_sgn.Cluster.unique()

In [ ]:
degs_day04_Differentiated_1 = degs_day04["Differentiated_1_day04"]
sgnPert_Differentiated_1 = df_all_04_sgn.loc[df_all_04_sgn.Cluster == "Differentiated_1",]
degs_day04_Differentiated_1=degs_day04_Differentiated_1.loc[degs_day04_Differentiated_1.target.isin(sgnPert_Differentiated_1.Perturbation),]
degs_day04_Differentiated_1 = degs_day04_Differentiated_1.loc[degs_day04_Differentiated_1.fdr < 0.05,]

degs_day04_Intermediate_1 = degs_day04["Intermediate_1_day04"]
sgnPert_Intermediate_1 = df_all_04_sgn.loc[df_all_04_sgn.Cluster == "Intermediate_1",]
degs_day04_Intermediate_1=degs_day04_Intermediate_1.loc[degs_day04_Intermediate_1.target.isin(sgnPert_Intermediate_1.Perturbation),]
degs_day04_Intermediate_1 = degs_day04_Intermediate_1.loc[degs_day04_Intermediate_1.fdr < 0.05,]


degs_day04_Neuroendocrine = degs_day04["Neuroendocrine_day04"]
sgnPert_Neuroendocrine = df_all_04_sgn.loc[df_all_04_sgn.Cluster == "Neuroendocrine",]
degs_day04_Neuroendocrine=degs_day04_Neuroendocrine.loc[degs_day04_Neuroendocrine.target.isin(sgnPert_Neuroendocrine.Perturbation),]
degs_day04_Neuroendocrine = degs_day04_Neuroendocrine.loc[degs_day04_Neuroendocrine.fdr < 0.05,]


In [ ]:
degs_day10_Differentiated_1 = degs_day10["Differentiated_1_day10"]
sgnPert_Differentiated_1 = df_all_10_sgn.loc[df_all_10_sgn.Cluster == "Differentiated_1",]
degs_day10_Differentiated_1=degs_day10_Differentiated_1.loc[degs_day10_Differentiated_1.target.isin(sgnPert_Differentiated_1.Perturbation),]
degs_day10_Differentiated_1 = degs_day10_Differentiated_1.loc[degs_day10_Differentiated_1.fdr < 0.05,]

degs_day10_Intermediate_1 = degs_day10["Intermediate_1_day10"]
sgnPert_Intermediate_1 = df_all_10_sgn.loc[df_all_10_sgn.Cluster == "Intermediate_1",]
degs_day10_Intermediate_1=degs_day10_Intermediate_1.loc[degs_day10_Intermediate_1.target.isin(sgnPert_Intermediate_1.Perturbation),]
degs_day10_Intermediate_1 = degs_day10_Intermediate_1.loc[degs_day10_Intermediate_1.fdr < 0.05,]


degs_day10_Neuroendocrine = degs_day10["Neuroendocrine_day10"]
sgnPert_Neuroendocrine = df_all_10_sgn.loc[df_all_10_sgn.Cluster == "Neuroendocrine",]
degs_day10_Neuroendocrine=degs_day10_Neuroendocrine.loc[degs_day10_Neuroendocrine.target.isin(sgnPert_Neuroendocrine.Perturbation),]
degs_day10_Neuroendocrine = degs_day10_Neuroendocrine.loc[degs_day10_Neuroendocrine.fdr < 0.05,]


In [ ]:
degs_day04_Differentiated_1.target=[x+ "_Differentiated_1_day04" for x in degs_day04_Differentiated_1.target]
degs_day04_Intermediate_1.target=[x+ "_Intermediate_1_day04" for x in degs_day04_Intermediate_1.target]
degs_day04_Neuroendocrine.target=[x+ "_Neuroendocrine_day04" for x in degs_day04_Neuroendocrine.target]


In [ ]:
degs_day10_Differentiated_1.target=[x+ "_Differentiated_1_day10" for x in degs_day10_Differentiated_1.target]
degs_day10_Intermediate_1.target=[x+ "_Intermediate_1_day10" for x in degs_day10_Intermediate_1.target]
degs_day10_Neuroendocrine.target=[x+ "_Neuroendocrine_day10" for x in degs_day10_Neuroendocrine.target]


In [ ]:
df_all = pd.concat([degs_day04_Differentiated_1,
                    degs_day04_Intermediate_1, 
                    degs_day04_Neuroendocrine,
                    degs_day10_Differentiated_1,
                    degs_day10_Intermediate_1,
                    degs_day10_Neuroendocrine], axis=0)

In [ ]:
df_all.to_csv("ALLSignDE.csv")

In [ ]:
for elem in df_all.target.unique():
    print(elem)
    tmp=df_all.loc[df_all.target==elem,:]
    print(list(tmp.feature))

In [ ]:
a=pd.DataFrame(df_all.feature.value_counts())
a
a=a.loc[a["count"]>4,]
a

In [ ]:
df_all_selected = df_all.loc[df_all.feature.isin(a.index),:]

In [ ]:
df_all_selected

In [ ]:
FCs = df_all_selected.pivot(index="target", columns="feature", values="fold_change")
FDRs = df_all_selected.pivot(index="target", columns="feature", values="fdr")
FCs = FCs.fillna(0)


In [ ]:
FCs.to_csv("ALLFCs.csv")

In [ ]:
len(list(FCs.columns))

In [ ]:
tmp = np.transpose(FCs.copy())
respAnnDat = sc.AnnData(X=tmp)
respAnnDat.obs["pertIndex"] = list(tmp.index)

sc.pp.scale(respAnnDat, max_value=10)
sc.pp.pca(respAnnDat, n_comps=50, svd_solver='arpack')
sc.pp.neighbors(respAnnDat, use_rep='X', n_neighbors=5)
sc.tl.leiden(respAnnDat, resolution=0.5)
sc.tl.umap(respAnnDat)
sc.pl.umap(respAnnDat, 
           color='leiden',
           size=15,  
           legend_fontoutline=3, 
           #legend_loc = 'center',
           legend_fontsize=14,
           legend_fontweight='normal')


In [ ]:
respAnnDat.obs["leiden"] = pd.Categorical(
    respAnnDat.obs["leiden"],
    categories=['0', '4', '5', '2', '1', '3','6','7','8'],
    ordered=True)


In [ ]:
FCs_day10.to_csv("FCs_day10.csv")
FDRs_day10.to_csv("FDRs_day10.csv")
FCs_day04.to_csv("FCs_day04.csv")
FDRs_day04.to_csv("FDRs_day04.csv")

In [ ]:
# FCs_day10[FDRs_day10>0.1]=0
# FCs_day04[FDRs_day04>0.1]=0

In [ ]:
df_meta = respAnnDat.obs.copy()
df_meta["gene_id"] = respAnnDat.obs_names.to_series().astype(str).values  # Flatten safely
df_meta = df_meta.sort_values(["leiden", "gene_id"])


In [ ]:
df_meta.to_csv("GeneClusters.csv")

In [ ]:
df_meta

In [ ]:
for elem in df_meta.leiden.unique():
    
    pathway = "Pathway " + str(elem)
    geneID = list(df_meta.loc[df_meta.leiden == elem,"gene_id"])
                        

    sc.tl.score_genes(adata=adata, gene_list=geneID, score_name=pathway)
    sc.pl.umap(adata, color=pathway, size=1, color_map="coolwarm")



In [ ]:
day04_degs = pd.read_csv("Day04DEGs.csv", index_col=0)

In [ ]:
day04_degs = day04_degs.loc[day04_degs.fdr<0.1,]

In [ ]:
a=pd.DataFrame(day04_degs.feature.value_counts())

In [ ]:
a.loc[a["count"]>5,:]

In [ ]:
day10_degs = pd.read_csv("Day04DEGs.csv", index_col=0)
day10_degs = day10_degs.loc[day10_degs.fdr<0.1,]

b=pd.DataFrame(day10_degs.feature.value_counts())
b_sign=b.loc[b["count"]>4,:]
day10_degs = day10_degs.loc[day10_degs.feature.isin(b_sign.index),:]
FCs = day10_degs.pivot(index="target", columns="feature", values="fold_change")
FDRs = day10_degs.pivot(index="target", columns="feature", values="fdr")
FCs = FCs.fillna(0)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
myTmp=FCs.copy()
myTmp[myTmp>2]=2
# mat: pandas DataFrame (rows/cols labeled) or 2D numpy array
g = sns.clustermap(
    myTmp,
    method="average",      # linkage: "single", "complete", "average", "ward"
    metric="euclidean",    # distance: "euclidean", "correlation", "cosine"
    standard_scale=None,   # or 0 to scale rows, 1 to scale cols
    z_score=None,          # or 0 to z-score rows, 1 to z-score cols
    cmap="vlag",           # change if you like
    figsize=(10, 10)
)

plt.show()


In [ ]:
tmp = np.transpose(FCs.copy())
respAnnDat = sc.AnnData(X=tmp)
respAnnDat.obs["pertIndex"] = list(tmp.index)

sc.pp.scale(respAnnDat, max_value=10)
sc.pp.pca(respAnnDat, n_comps=5, svd_solver='arpack')
sc.pp.neighbors(respAnnDat, use_rep='X', n_neighbors=5)
sc.tl.leiden(respAnnDat, resolution=0.5)
sc.tl.umap(respAnnDat)
sc.pl.umap(respAnnDat, 
           color='leiden',
           size=15,  
           legend_fontoutline=3, 
           #legend_loc = 'center',
           legend_fontsize=14,
           legend_fontweight='normal')
